# Pipeline medalhão
Criação das tabelas seguindo as camadas da arquitetura medalhão 

## Definindo variaveis

In [0]:

CATALOG = "superstore_sales"
volume_path = "/Volumes/superstore_sales/bronze/external_files/"
bronze_table = "bronze.train"
silver_table = "silver.sales"

## 1.Camada bronze
Ingerindo a partir do arquivo train.csv, sem tratamentos. 

In [0]:

# Lê o CSV usando read_files e adiciona colunas de metadados
df = spark.sql(f"""
SELECT
     *
    ,_metadata.file_path AS _ingestion_file_path
    ,current_timestamp() AS _ingestion_ts
FROM read_files("{volume_path}", format => 'csv', header => true, encoding => 'ISO-8859-1')
""")

# Renomeia colunas substituindo espaços por underscores
new_columns = [c.replace(" ", "_") for c in df.columns]
df = df.toDF(*new_columns)

# Cria a tabela bronze
df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{bronze_table}")
print(f"Tabela {CATALOG}.{bronze_table} criada com {df.count()} registros.")

## 2. Camada silver
Transformação de tipos, nome de campos com snack_case e datas formatadas

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{silver_table} 
SELECT 
     Row_ID AS row_id
    ,Order_ID AS order_id
    ,TRY_CAST(Order_Date AS date) AS order_date
    ,TRY_CAST(Ship_Date AS date) AS ship_date
    ,Ship_Mode AS ship_mode
    ,Customer_ID AS custumer_id
    ,Customer_Name AS custumer_name
    ,Segment AS segment
    ,Country AS country
    ,City AS city
    ,State AS state
    ,Postal_Code AS postal_code
    ,Region AS region
    ,Product_ID AS product_id
    ,Category AS category
    ,`Sub-Category` AS subcategory
    ,Product_Name AS product_name
    ,TRY_CAST(Sales AS decimal(18,2)) AS sales
FROM {CATALOG}.{bronze_table}
WHERE TRY_CAST(Sales AS decimal(18,2)) > 0
""")